# Denton CAD — 2026 Appraisal Export ETL

Parses every `.TXT` file in each Denton export folder listed in `EXPORT_DIRS` and writes one CSV per file to `output/` (the `exportTotals.pdf` and any other non-`.TXT` files are ignored):
- `Data/2026 CERTIFIED DATA - ALL PROPERTIES/` — 2026-07-27 certified export (supplement 0)
- `Data/2026-3/` — 2026-10-14 export (supplement 3)

Output names keep the source's date prefix (e.g. `2026-10-14_2026_APPRAISAL_INFO.csv`), so exports never overwrite each other.

**Source format** (checked on the 2026-07-27 and 2026-10-14 exports — identical layout, PACS 8.1.33 / export version 8.0.0.33):
- This is True Automation's PACS **Appraisal Export** — *not* the pipe-delimited layout Tarrant uses
- **Most files are fixed-width** (every line the same length, no delimiter), CRLF line endings, no header row
- Only `APPRAISAL_SB12.TXT` and `APPRAISAL_SKETCH_INFO.TXT` are **tab-separated** (no header, trailing tab on every line)
- Field positions come from `pacs_appraisal_export_layout.json` (PACS layout 8.0.30). Denton's 8.0.33 records are wider than 8.0.30 for
  `APPRAISAL_INFO` (9922 vs 9247), `APPRAISAL_ENTITY_INFO` (3141 vs 2750) and `APPRAISAL_ENTITY_TOTALS` (2491 vs 2140) —
  the extra fields are appended at the end, so the known fields still line up and the leftover bytes are kept in an `unmapped_tail` column
- Numbers are zero-padded strings (e.g. `000000000011`, `02026`); some carry implied decimals (e.g. `size_acres`, 4 decimals). Values are kept as text here

Steps: **Inventory → Parse (stream line by line) → Export CSV → Validate**.

## 1. Setup

In [1]:
import csv
import json
import re
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_ROOT = Path("../Data")
EXPORT_DIRS = [                     # add new Denton export folders here
    DATA_ROOT / "2026 CERTIFIED DATA - ALL PROPERTIES",
    DATA_ROOT / "2026-3",
]
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

LAYOUT_FILE = Path("pacs_appraisal_export_layout.json")
LAYOUT = json.loads(LAYOUT_FILE.read_text())

SKIP_EXISTING = False   # True = don't re-parse files whose CSV is already in output/ (the INFO / ENTITY_INFO files are several GB)

for d in EXPORT_DIRS:
    print(d.resolve())
print(f"{len(LAYOUT)} fixed-width record layouts loaded")

/Users/user2/PTP/CAD Website/Denton/Data/2026 CERTIFIED DATA - ALL PROPERTIES
18 fixed-width record layouts loaded


## 2. Tab-separated files
These two aren't in the PACS layout. Column names were worked out from the data by matching IDs against the other files
(e.g. `imprv_id` / `imprv_det_id` match `APPRAISAL_IMPROVEMENT_DETAIL`). SB12 columns that couldn't be identified with confidence are left as `col_NN`.

In [2]:
TAB_COLUMNS = {
    "APPRAISAL_SKETCH_INFO": [
        "prop_id", "prop_val_yr", "imprv_id", "imprv_det_id",
        "sketch_type",   # Legacy / Enhanced
        "sketch_cmds",   # Legacy sketch command string, e.g. U33,R48,D33
        "sketch_json",   # Enhanced sketch as JSON
        "_trailing",
    ],
    "APPRAISAL_SB12": [
        "prop_id", "owner_id", "entity_id", "entity_cd", "entity_name",
        "tea_district_num",   # e.g. 061-902-02
        "exmpt_type_cd", "qualify_yr", "seq_num", "tax_yr", "col_11",
        "record_type",        # SB12 / SUMMARY / NEW_VALUE
        "col_13", "col_14", "col_15", "col_16", "col_17", "col_18",
        "prop_val_yr",
        "_trailing",
    ],
}

## 3. Inventory

In [3]:
def record_key(path):
    """2026-07-27_2026_APPRAISAL_INFO.TXT -> APPRAISAL_INFO"""
    return re.sub(r"^\d{4}-\d{2}-\d{2}_\d+_", "", path.stem)


def first_line(path):
    with open(path, "rb") as f:
        return f.readline().rstrip(b"\r\n").decode("latin-1")


rows = []
for path in (p for d in EXPORT_DIRS for p in sorted(d.glob("*.TXT"))):
    key = record_key(path)
    line = first_line(path)
    fmt = "tab" if "\t" in line else "fixed"
    layout_width = LAYOUT[key][-1][2] if key in LAYOUT else None
    rows.append({
        "export": path.parent.name,
        "path": path,
        "file": path.name,
        "key": key,
        "format": fmt,
        "size_mb": round(path.stat().st_size / 1e6, 1),
        "record_width": len(line) if fmt == "fixed" else None,
        "layout_width": layout_width,
        "has_layout": key in LAYOUT or key in TAB_COLUMNS,
    })

inventory = pd.DataFrame(rows)
inventory.drop(columns="path")

,file,key,format,size_mb,record_width,layout_width,has_layout
0,2026-07-27_2026_APPRAISAL_ABSTRACT_SUBDV.TXT,APPRAISAL_ABSTRACT_SUBDV,fixed,0.6,50.0,50.0,True
1,2026-07-27_2026_APPRAISAL_AGENT.TXT,APPRAISAL_AGENT,fixed,0.9,378.0,378.0,True
2,2026-07-27_2026_APPRAISAL_ARB.TXT,APPRAISAL_ARB,fixed,0.6,127.0,127.0,True
3,2026-07-27_2026_APPRAISAL_COUNTRY_CODE.TXT,APPRAISAL_COUNTRY_CODE,fixed,0.0,55.0,55.0,True
4,2026-07-27_2026_APPRAISAL_ENTITY.TXT,APPRAISAL_ENTITY,fixed,0.0,17.0,17.0,True
5,2026-07-27_2026_APPRAISAL_ENTITY_INFO.TXT,APPRAISAL_ENTITY_INFO,fixed,6311.5,3141.0,2750.0,True
6,2026-07-27_2026_APPRAISAL_ENTITY_TOTALS.TXT,APPRAISAL_ENTITY_TOTALS,fixed,0.7,2491.0,2140.0,True
7,2026-07-27_2026_APPRAISAL_HEADER.TXT,APPRAISAL_HEADER,fixed,0.0,244.0,244.0,True
8,2026-07-27_2026_APPRAISAL_IMPROVEMENT_DETAIL.TXT,APPRAISAL_IMPROVEMENT_DETAIL,fixed,972.3,622.0,622.0,True
9,2026-07-27_2026_APPRAISAL_IMPROVEMENT_DETAIL_A...,APPRAISAL_IMPROVEMENT_DETAIL_ATTR,fixed,248.0,87.0,87.0,True


In [4]:
# Every file must have a known layout before we parse
missing = inventory.loc[~inventory["has_layout"], "file"].tolist()
assert not missing, f"No layout for: {missing}"

# Fixed-width records shorter than the layout would mean the layout is wrong for this export
too_short = inventory.query("format == 'fixed' and record_width < layout_width")
assert too_short.empty, too_short
inventory.query("format == 'fixed' and record_width > layout_width")   # these get an unmapped_tail column

,file,key,format,size_mb,record_width,layout_width,has_layout
5,2026-07-27_2026_APPRAISAL_ENTITY_INFO.TXT,APPRAISAL_ENTITY_INFO,fixed,6311.5,3141.0,2750.0,True
6,2026-07-27_2026_APPRAISAL_ENTITY_TOTALS.TXT,APPRAISAL_ENTITY_TOTALS,fixed,0.7,2491.0,2140.0,True
11,2026-07-27_2026_APPRAISAL_INFO.TXT,APPRAISAL_INFO,fixed,4795.2,9922.0,9247.0,True


## 4. Parsers
Both stream the file line by line and write straight to CSV, so the multi-GB files never sit in memory.
Files are read as bytes and split on `\n` only, so a stray `\r` inside a value can't break a record; `latin-1` decoding never fails.

In [5]:
def parse_fixed_width(path, layout, out_path):
    """Slice each line by the layout positions, strip padding, write CSV. Returns (rows, bad_length_rows)."""
    record_width = len(first_line(path))
    layout_width = layout[-1][2]
    names = [name for name, _, _ in layout]
    slices = [(start - 1, end) for _, start, end in layout]
    if record_width > layout_width:
        names.append("unmapped_tail")
        slices.append((layout_width, None))

    n = bad = 0
    with open(path, "rb") as src, open(out_path, "w", newline="", encoding="utf-8") as dst:
        writer = csv.writer(dst)
        writer.writerow(names)
        for raw_line in src:
            line = raw_line.rstrip(b"\r\n").decode("latin-1")
            if len(line) != record_width:
                bad += 1
                if bad <= 5:
                    print(f"  BAD LENGTH line {n + 1}: {len(line)} chars (expected {record_width})")
            writer.writerow([line[s:e].strip() for s, e in slices])
            n += 1
    return n, bad


def parse_tab(path, columns, out_path):
    """Tab-separated, no header. Returns (rows, bad_field_count_rows)."""
    n = bad = 0
    with open(path, "rb") as src, open(out_path, "w", newline="", encoding="utf-8") as dst:
        writer = csv.writer(dst)
        writer.writerow([c for c in columns if c != "_trailing"])
        for raw_line in src:
            fields = [f.strip() for f in raw_line.rstrip(b"\r\n").decode("latin-1").split("\t")]
            if len(fields) != len(columns):
                bad += 1
                if bad <= 5:
                    print(f"  BAD FIELD COUNT line {n + 1}: {len(fields)} fields (expected {len(columns)})")
                fields = (fields + [""] * len(columns))[: len(columns)]
            writer.writerow([v for c, v in zip(columns, fields) if c != "_trailing"])
            n += 1
    return n, bad

## 5. Parse → CSV

In [6]:
results = []
for rec in inventory.itertuples():
    path = rec.path
    out_path = OUTPUT_DIR / f"{path.stem}.csv"
    if SKIP_EXISTING and out_path.exists():
        print(f"skip  {rec.file}")
        continue

    print(f"parse {rec.file} ({rec.size_mb:,.1f} MB, {rec.format})")
    t0 = time.time()
    if rec.format == "tab":
        n, bad = parse_tab(path, TAB_COLUMNS[rec.key], out_path)
    else:
        n, bad = parse_fixed_width(path, LAYOUT[rec.key], out_path)
    results.append({
        "export": rec.export,
        "key": rec.key,
        "file": rec.file,
        "output": out_path.name,
        "rows": n,
        "bad_rows": bad,
        "out_mb": round(out_path.stat().st_size / 1e6, 1),
        "seconds": round(time.time() - t0, 1),
    })

summary = pd.DataFrame(results)
summary.drop(columns="key")

parse 2026-07-27_2026_APPRAISAL_ABSTRACT_SUBDV.TXT (0.6 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_AGENT.TXT (0.9 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_ARB.TXT (0.6 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_COUNTRY_CODE.TXT (0.0 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_ENTITY.TXT (0.0 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_ENTITY_INFO.TXT (6,311.5 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_ENTITY_TOTALS.TXT (0.7 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_HEADER.TXT (0.0 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_IMPROVEMENT_DETAIL.TXT (972.3 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_IMPROVEMENT_DETAIL_ATTR.TXT (248.0 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_IMPROVEMENT_INFO.TXT (48.5 MB, fixed)
parse 2026-07-27_2026_APPRAISAL_INFO.TXT (4,795.2 MB, fixed)


KeyboardInterrupt: 

## 6. Validate

In [ ]:
# Row counts in the CSVs must match the source line counts, and no malformed lines
assert (summary["bad_rows"] == 0).all(), summary.query("bad_rows > 0")
print(f"{summary['rows'].sum():,} rows written across {len(summary)} files")

In [ ]:
# Row counts per file, side by side across exports
summary.pivot(index="key", columns="export", values="rows")

In [ ]:
def preview(key, export_date=None, n=5):
    """Head of one output CSV; newest export by default, or e.g. export_date="2026-07-27"."""
    out_path = sorted(OUTPUT_DIR.glob(f"{export_date or '*'}_*_{key}.csv"))[-1]
    return pd.read_csv(out_path, dtype=str, keep_default_na=False, nrows=n)

preview("APPRAISAL_INFO")

In [ ]:
preview("APPRAISAL_IMPROVEMENT_DETAIL")

In [ ]:
preview("APPRAISAL_LAND_DETAIL")

In [ ]:
preview("APPRAISAL_SKETCH_INFO")

In [ ]:
preview("APPRAISAL_SB12")